# 00 · Helpers (fungsi bersama)

Notebook ini **tidak perlu dijalankan sendiri**. Setiap notebook lain memanggilnya di awal dengan:

```python
%run ./00_helpers.ipynb
```

Isinya hal-hal yang **harus sama untuk semua model**, sesuai *Prinsip Evaluasi Terstandarisasi* di panduan tugas:
- dataset dan split **80/10/10** yang sama,
- **random seed tetap 42**,
- `batch_size = 128`,
- cara menghitung metrik (MSE, PSNR, SSIM, FID, latency) yang sama.

Jadi kalau hasil antar model berbeda, penyebabnya memang dari model dan loss function-nya, bukan dari perbedaan cara mengolah data.

## 1. Import Library

In [ ]:
import os, json, time, random, copy, math, csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.stats import rankdata
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm

## 2. Konfigurasi Umum

- `QUICK = True` → **tes cepat**: dataset asli tetapi hanya sebagian kecil, 1 epoch. Pakai ini dulu untuk memastikan semua notebook jalan tanpa error. Hasilnya disimpan terpisah di `results_quick/` dan **tidak untuk laporan**.
- `QUICK = False` → **eksperimen penuh** untuk laporan.

In [ ]:
QUICK = False          # ubah ke True untuk tes cepat

DATASET = "fashion_mnist"          # fashion_mnist | mnist | kmnist
SEED = 42
BATCH_SIZE = 128                   # panduan: 128 atau 256
SPLIT = (0.8, 0.1, 0.1)            # train / val / test
IMG_SHAPE = (1, 28, 28)
IMG_SIZE = 28 * 28                 # 784

FID_NUM_SAMPLES = 256 if QUICK else 5000   # jumlah sampel sintetis per model untuk FID
LATENCY_BATCH = 64                         # panduan: latency per batch 64 sampel
INTERP_CLASSES = (1, 8)                    # Figur 4: Trouser -> Bag (dua kelas kontras)
ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# Folder proyek = folder yang berisi requirements.txt (notebook ada di subfolder src/)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), Path.cwd())
DATA_DIR = ROOT / "data"                                               # dataset (otomatis diunduh)
CKPT_DIR = ROOT / ("checkpoints_quick" if QUICK else "checkpoints")    # bobot model terlatih
RESULTS_DIR = ROOT / ("results_quick" if QUICK else "results")         # semua hasil untuk laporan
TABLES_DIR = RESULTS_DIR / "tables"
for p in [CKPT_DIR, RESULTS_DIR / "metrics", RESULTS_DIR / "artifacts",
          RESULTS_DIR / "histories", RESULTS_DIR / "figures", TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def n_epochs(n):
    # jumlah epoch; otomatis 1 di mode QUICK
    return 1 if QUICK else n

print(f"device = {device} | dataset = {DATASET} | QUICK = {QUICK}")
print(f"folder proyek = {ROOT}\nhasil -> {RESULTS_DIR}")

## 3. Utilitas: seed, konversi rentang piksel, simpan/muat

In [ ]:
def set_seed(seed=SEED):
    # Seed tetap: torch.manual_seed(42), np.random.seed(42), random.seed(42)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def to_pm1(x):
    # [0, 1] -> [-1, 1]  (input GAN / DCGAN / DDPM)
    return x * 2.0 - 1.0

def to_01(x):
    # [-1, 1] -> [0, 1]  (output Tanh dikembalikan ke rentang piksel biasa)
    return ((x + 1.0) / 2.0).clamp(0.0, 1.0)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def save_results(name, metrics, artifacts=None, history=None):
    # Menyimpan hasil satu model agar bisa dikumpulkan di notebook 09.
    save_json(metrics, RESULTS_DIR / "metrics" / f"{name}.json")
    if artifacts is not None:
        torch.save(artifacts, RESULTS_DIR / "artifacts" / f"{name}.pt")
    if history is not None:
        save_json(history, RESULTS_DIR / "histories" / f"{name}.json")
    print(f"Hasil '{name}' tersimpan di {RESULTS_DIR}/")

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

## 4. Dataset & Split 80/10/10

- Train set (60.000) dan test set (10.000) bawaan torchvision **digabung** → 70.000 citra.
- Diacak dengan `torch.Generator().manual_seed(42)`, lalu dibagi **56.000 train / 7.000 val / 7.000 test**.
- Piksel dinormalisasi ke **[0, 1]** (dibagi 255).
- DataLoader: `shuffle=True` untuk train, `shuffle=False` untuk val/test.

In [ ]:
CLASS_NAMES = {
    "fashion_mnist": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                      "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
    "mnist": [str(i) for i in range(10)],
    "kmnist": ["o", "ki", "su", "tsu", "na", "ha", "ma", "ya", "re", "wo"],
}[DATASET]

def _load_all():
    cls = {"fashion_mnist": datasets.FashionMNIST, "mnist": datasets.MNIST,
           "kmnist": datasets.KMNIST}[DATASET]
    tr = cls(DATA_DIR, train=True, download=True)
    te = cls(DATA_DIR, train=False, download=True)
    x = torch.cat([tr.data, te.data]).float().div(255.0).unsqueeze(1)    # (70000, 1, 28, 28)
    y = torch.cat([torch.as_tensor(tr.targets), torch.as_tensor(te.targets)]).long()
    return x, y

def load_splits():
    x, y = _load_all()
    n = len(x)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    n_tr, n_va = int(SPLIT[0] * n), int(SPLIT[1] * n)
    idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}
    if QUICK:
        idx = {"train": idx["train"][:2048], "val": idx["val"][:512], "test": idx["test"][:512]}
    splits = {k: (x[v], y[v]) for k, v in idx.items()}
    info = {k: int(len(v)) for k, v in idx.items()}
    save_json({**info, "total": int(n), "dataset": DATASET, "seed": SEED}, RESULTS_DIR / "split_info.json")
    print("Jumlah data:", info)
    return splits

def make_loaders(splits):
    loaders = {}
    for name, (x, y) in splits.items():
        loaders[name] = DataLoader(
            TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=(name == "train"),
            generator=torch.Generator().manual_seed(SEED) if name == "train" else None)
    return loaders

def fixed_examples(splits, n=8):
    # 8 citra uji tetap (1 per kelas 0..7) -> dipakai di Figur 1 & Figur 3
    x, y = splits["test"]
    idx = torch.stack([torch.nonzero(y == c)[0, 0] for c in range(n)])
    return x[idx], y[idx]

def interp_endpoints(splits):
    # Citra uji pertama dari dua kelas kontras -> titik z_A dan z_B (Figur 4)
    x, y = splits["test"]
    a, b = INTERP_CLASSES
    return x[y == a][:1], x[y == b][:1]

## 5. Metrik Rekonstruksi: MSE, PSNR, SSIM

- **MSE**: rata-rata selisih kuadrat per piksel. Makin kecil, makin mirip.
- **PSNR** = 10·log10(1 / MSE), dalam dB. Makin besar, makin bagus.
- **SSIM**: kemiripan *struktur* (pola terang-gelap, kontras), nilainya 0–1. Makin besar, makin bagus.

In [ ]:
def per_image_mse(x_hat, x):
    return ((x_hat - x) ** 2).flatten(1).mean(1)

def psnr_from_mse(mse):
    return 10.0 * torch.log10(1.0 / mse.clamp_min(1e-10))

def ssim_batch(x_hat, x):
    a = x_hat.detach().cpu().numpy()
    b = x.detach().cpu().numpy()
    return np.array([structural_similarity(b[i, 0], a[i, 0], data_range=1.0, gaussian_weights=True,
                                           sigma=1.5, use_sample_covariance=False)
                     for i in range(len(a))])

@torch.no_grad()
def reconstruction_metrics(recon_fn, x_test, bs=500):
    # recon_fn: citra [0,1] -> rekonstruksi [0,1]
    mses, ssims = [], []
    for i in range(0, len(x_test), bs):
        xb = x_test[i:i + bs].to(device)
        xh = recon_fn(xb).clamp(0, 1)
        mses.append(per_image_mse(xh, xb).cpu())
        ssims.append(ssim_batch(xh, xb))
    mse = torch.cat(mses)
    psnr = psnr_from_mse(mse)
    ssim = np.concatenate(ssims)
    return {"mse": float(mse.mean()), "mse_std": float(mse.std()),
            "psnr": float(psnr.mean()), "psnr_std": float(psnr.std()),
            "ssim": float(ssim.mean()), "ssim_std": float(ssim.std())}

@torch.no_grad()
def apply_batched(fn, x, bs=500):
    return torch.cat([fn(x[i:i + bs].to(device)).cpu() for i in range(0, len(x), bs)])

## 6. FID (Fréchet Inception Distance)

FID membandingkan **sebaran** gambar buatan dengan **sebaran** gambar asli, menggunakan fitur dari jaringan InceptionV3. Makin kecil FID, makin mirip, baik dari segi kualitas maupun keberagaman.

- Gambar grayscale 1 kanal diulang jadi 3 kanal, lalu diperbesar ke 299×299.
- Statistik gambar asli dihitung sekali dari seluruh Test Set, lalu disimpan (cache).
- Setiap model generatif dinilai dengan jumlah sampel yang sama (`FID_NUM_SAMPLES`).

In [ ]:
_FID_MODEL = None

def _fid_model():
    global _FID_MODEL
    if _FID_MODEL is None:
        from pytorch_fid.inception import InceptionV3
        _FID_MODEL = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]],
                                 resize_input=True, normalize_input=True).to(device).eval()
    return _FID_MODEL

@torch.no_grad()
def inception_features(images, bs=32):
    # bs kecil (32) supaya hemat memori GPU (RTX 4050 6 GB).
    m = _fid_model()
    dev = next(m.parameters()).device
    feats = []
    for i in range(0, len(images), bs):
        x = images[i:i + bs].to(dev).float().repeat(1, 3, 1, 1)
        feats.append(m(x)[0].squeeze(-1).squeeze(-1).cpu())
    return torch.cat(feats).numpy().astype(np.float64)

def _free_gpu_memory():
    import gc
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

def safe_inception_features(images):
    # Kalau memori GPU tidak cukup: kosongkan cache, coba batch lebih kecil, terakhir pindah ke CPU.
    global _FID_MODEL
    _free_gpu_memory()
    for bs in (32, 8):
        try:
            return inception_features(images, bs)
        except torch.cuda.OutOfMemoryError:
            print(f"[INFO] Memori GPU tidak cukup untuk FID (batch {bs}), mencoba cara lain...")
            _free_gpu_memory()
    print("[INFO] FID dihitung memakai CPU (lebih lambat, hasilnya sama).")
    _FID_MODEL = _fid_model().cpu()
    try:
        return inception_features(images, 32)
    finally:
        if device.type == "cuda":
            _FID_MODEL = _FID_MODEL.to(device)

def _sqrtm(m):
    out = linalg.sqrtm(m)
    return out[0] if isinstance(out, tuple) else out

def frechet_distance(mu1, s1, mu2, s2, eps=1e-6):
    diff = mu1 - mu2
    covmean = _sqrtm(s1.dot(s2))
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * eps
        covmean = _sqrtm((s1 + off).dot(s2 + off))
    covmean = covmean.real
    return float(diff.dot(diff) + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))

def real_fid_stats(splits):
    cache = CKPT_DIR / "fid_real_stats.npz"     # cache statistik Inception citra asli
    if cache.exists():
        d = np.load(cache)
        return d["mu"], d["sigma"]
    f = safe_inception_features(splits["test"][0])
    mu, sigma = f.mean(0), np.cov(f, rowvar=False)
    np.savez(cache, mu=mu, sigma=sigma)
    return mu, sigma

def compute_fid(gen_images, splits):
    try:
        mu_r, s_r = real_fid_stats(splits)
        f = safe_inception_features(gen_images)
        return frechet_distance(mu_r, s_r, f.mean(0), np.cov(f, rowvar=False))
    except Exception as e:
        print(f"[PERINGATAN] FID gagal dihitung: {e}")
        return None

## 7. Latency Inferensi

Waktu rata-rata (milidetik) untuk menghasilkan **64 sampel** (sesuai panduan) dan **1 sampel** (untuk analisis aplikasi real-time di Q6). Model dijalankan beberapa kali dulu sebagai pemanasan (*warm-up*) supaya waktunya stabil.

In [ ]:
@torch.no_grad()
def measure_latency(fn, warmup=5, runs=30):
    if QUICK:
        warmup, runs = 1, 3
    for _ in range(warmup):
        fn()
    sync()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        sync()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

## 8. Mode Coverage (deteksi *mode collapse*)

Gambar buatan setiap model diklasifikasikan memakai **MLP Classifier dari notebook 01**. Kalau model generatifnya sehat, ke-10 kelas akan muncul kurang lebih merata (masing-masing sekitar 10%). Kalau terjadi *mode collapse*, hanya sedikit kelas yang muncul.

- `entropy_norm`: 1.0 berarti merata sempurna, mendekati 0 berarti *collapse*.
- `modes_covered`: jumlah kelas yang porsinya minimal 2%.

In [ ]:
def load_mlp_classifier():
    p = CKPT_DIR / "mlp_classifier_traced.pt"
    if not p.exists():
        print("[INFO] MLP Classifier belum ada. Jalankan notebook 01 dulu agar mode coverage bisa dihitung.")
        return None
    return torch.jit.load(str(p), map_location=device).eval()

@torch.no_grad()
def mode_coverage(images, clf):
    if clf is None:
        return None
    preds = apply_batched(lambda x: clf(x).argmax(1), images)
    p = torch.bincount(preds, minlength=10).float() / len(preds)
    nz = p[p > 0]
    return {"class_hist": p.tolist(),
            "entropy_norm": max(0.0, float(-(nz * nz.log()).sum() / np.log(10))),
            "modes_covered": int((p >= 0.02).sum())}

## 9. Deteksi Anomali (pendukung Q6 bagian 1)

Pada citra uji ditambahkan **cacat buatan** seperti goresan atau noda, mirip cacat pada inspeksi manufaktur. Autoencoder yang baik akan menghasilkan *reconstruction error* yang **lebih besar** pada citra cacat. Kemampuan memisahkan citra normal dan citra cacat diukur dengan **AUROC**: 0.5 berarti sama dengan menebak acak, 1.0 berarti sempurna.

In [ ]:
def add_synthetic_defects(x, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    x = x.clone()
    n, _, h, w = x.shape
    for i in range(n):
        if torch.rand(1, generator=g).item() < 0.5:          # goresan
            r = int(torch.randint(4, h - 4, (1,), generator=g))
            c0 = int(torch.randint(2, w // 2, (1,), generator=g))
            length = int(torch.randint(8, w // 2 + 4, (1,), generator=g))
            for k in range(length):
                rr, cc = min(h - 1, r + k // 3), min(w - 1, c0 + k)
                x[i, :, rr, cc] = 1.0 - x[i, :, rr, cc]
        else:                                                # noda
            s = int(torch.randint(3, 6, (1,), generator=g))
            r = int(torch.randint(4, h - s - 4, (1,), generator=g))
            c = int(torch.randint(4, w - s - 4, (1,), generator=g))
            x[i, :, r:r + s, c:c + s] = float(torch.rand(1, generator=g))
    return x

def auroc(scores_normal, scores_anomaly):
    s = np.concatenate([scores_normal, scores_anomaly])
    ranks = rankdata(s)
    n0, n1 = len(scores_normal), len(scores_anomaly)
    return float((ranks[n0:].sum() - n1 * (n1 + 1) / 2) / (n0 * n1))

@torch.no_grad()
def anomaly_auroc(recon_fn, x_test):
    x_def = add_synthetic_defects(x_test)
    err = lambda x: apply_batched(lambda b: per_image_mse(recon_fn(b), b), x).numpy()
    e_norm, e_def = err(x_test), err(x_def)
    return {"auroc": auroc(e_norm, e_def), "err_normal_mean": float(e_norm.mean()),
            "err_defect_mean": float(e_def.mean()),
            "err_ratio": float(e_def.mean() / max(e_norm.mean(), 1e-12))}

## 10. Ringkasan Kestabilan Training

Ukuran sederhana: **koefisien variasi** (simpangan baku ÷ rata-rata) loss pada paruh akhir training. Nilai besar berarti loss masih naik-turun (berosilasi). Ini hanya heuristik, jadi tetap cek dengan melihat grafik loss.

In [ ]:
def stability_summary(history):
    def cv(a):
        a = np.asarray(a, float)
        t = a[len(a) // 2:] if len(a) >= 4 else a
        return float(np.std(t) / (abs(np.mean(t)) + 1e-12))
    if "iter_g_loss" in history:
        out = {"g_loss_cv_last_half": cv(history["iter_g_loss"]),
               "d_loss_cv_last_half": cv(history["iter_d_loss"]),
               "final_D_x": history["train"][-1]["D_x"], "final_D_G_z": history["train"][-1]["D_G_z"]}
        out["label"] = "Rentan osilasi" if out["g_loss_cv_last_half"] > 0.25 else "Cukup stabil"
    else:
        out = {"loss_cv_last_half": cv([e["loss"] for e in history["train"]])}
        out["label"] = "Stabil / Konvergen" if out["loss_cv_last_half"] < 0.25 else "Berfluktuasi"
    return out

## 11. Interpolasi Laten (pendukung Figur 4 dan Q4)

Untuk setiap α, dicatat:
- `nn_dist_ratio`: jarak z(α) ke kode laten data latih yang paling dekat, dibagi jarak "normal" antar data latih. **Nilai > 1 berarti z(α) masuk ke "wilayah kosong" (*latent void*)**, yaitu daerah yang tidak pernah dilihat decoder saat training.
- `classifier_max_conf`: seberapa yakin MLP Classifier bahwa gambar hasil decode termasuk suatu kelas. Nilai rendah berarti gambarnya tidak mirip pakaian apa pun.

In [ ]:
@torch.no_grad()
def nn_distance(z, bank, bs=256):
    return torch.cat([torch.cdist(z[i:i + bs], bank).min(1).values for i in range(0, len(z), bs)])

@torch.no_grad()
def interpolation_record(name, za, zb, decode, clf=None, bank=None):
    # za, zb: (1, d) di device. decode: z -> citra [0,1]. bank: kode laten data latih (opsional)
    zs = torch.cat([(1 - a) * za + a * zb for a in ALPHAS])
    imgs = decode(zs).cpu()
    conf = F.softmax(clf(imgs.to(device)), 1).max(1).values.cpu() if clf is not None else None
    ratio = None
    if bank is not None:
        ref = nn_distance(bank[:1000], bank[1000:]).median()
        ratio = (nn_distance(zs, bank) / ref).cpu()
    rows = []
    for i, a in enumerate(ALPHAS):
        rows.append({"model": name, "alpha": a,
                     "nn_dist_ratio": None if ratio is None else float(ratio[i]),
                     "classifier_max_conf": None if conf is None else float(conf[i]),
                     "pixel_change_from_prev": 0.0 if i == 0 else float((imgs[i] - imgs[i - 1]).abs().mean())})
    return imgs, rows

## 12. Analisis Variasi Intra-Kelas & Sampling Gaussian (pendukung Q4, Q5)

- `intra_class_analysis`: apakah rekonstruksi masih mempertahankan **variasi di dalam satu kelas**?
  - `var_ratio` ≈ 1 berarti variasi terjaga. Mendekati 0 berarti semua gambar berubah jadi "gambar rata-rata kelas".
  - `closer_to_class_mean`: proporsi rekonstruksi yang lebih mirip rata-rata kelas dibanding gambar aslinya sendiri.
- `gaussian_fit_sampling`: mengambil sampel z ~ N(μ_z, diag σ_z²) dari statistik kode laten citra uji.

In [ ]:
@torch.no_grad()
def intra_class_analysis(recon, x, y):
    ratios, closer = [], []
    for c in range(10):
        m = y == c
        if m.sum() < 2:
            continue
        xc, rc = x[m].flatten(1), recon[m].flatten(1)
        ratios.append((rc.var(0).mean() / xc.var(0).mean().clamp_min(1e-12)).item())
        mean_c = xc.mean(0, keepdim=True)
        closer.append((((rc - mean_c) ** 2).mean(1) < ((rc - xc) ** 2).mean(1)).float())
    return {"var_ratio": float(np.mean(ratios)), "closer_to_class_mean": float(torch.cat(closer).mean())}

def gaussian_fit_sampling(z_test, n, seed=SEED):
    mu, sigma = z_test.mean(0), z_test.std(0)
    z = mu + sigma * torch.randn(n, z_test.size(1), generator=torch.Generator().manual_seed(seed))
    return z, {"frac_negative_entries": float((z < 0).float().mean()),
               "frac_zero_entries_real": float((z_test <= 1e-6).float().mean())}

## 13. Gaya Plot & Tampilan Grid Gambar

In [ ]:
COLORS = {"vanilla_ae": "#2a78d6", "sparse_ae": "#eb6834", "vae": "#1baf7a",
          "gan": "#eda100", "dcgan": "#e87ba4", "ddpm": "#008300"}
SHORT = {"vanilla_ae": "Vanilla AE", "sparse_ae": "Sparse AE", "vae": "VAE",
         "gan": "Vanilla GAN", "dcgan": "DCGAN", "ddpm": "DDPM"}
plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.grid": True, "grid.color": "#e4e3de",
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "lines.linewidth": 1.6, "savefig.bbox": "tight"})

def grid_img(imgs, nrow=None):
    g = make_grid(imgs.clamp(0, 1), nrow=nrow or len(imgs), padding=2, pad_value=1.0)
    return g.permute(1, 2, 0).numpy()

def show_imgs(ax, imgs, nrow=None, label=None, title=None):
    ax.imshow(grid_img(imgs, nrow), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    if label:
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
    if title:
        ax.set_title(title, loc="left")

def save_fig(fig, name):
    fig.savefig(RESULTS_DIR / "figures" / f"{name}.png", dpi=300)
    fig.savefig(RESULTS_DIR / "figures" / f"{name}.pdf")
    print(f"Figur tersimpan: {RESULTS_DIR / 'figures' / name}.png")

print("Helpers siap ✔")